# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maqsood-Ahmed110/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

Two paper findings + methodology questions

Finding I'm choosing #1: "Growth Prediction" model (Part IV)

Claims 90% accuracy on same-brand pages, 75% on unseen brands, trained on 96.6K pages.

My methodology question: Where does the "growing vs declining" label actually come from, and over what time window? The paper doesn't specify whether the label is defined using a fixed percentage threshold (like my own impressions < median proxy) or something else, and critically — is the label observed strictly after the feature window, or could there be overlap? If "days visible" (their #1 predictor) is measured over a window that overlaps the labeling window, that's a real leakage risk, the same trap I deliberately created and removed in my own w03 notebook. The paper reports a same-brand vs. new-brand accuracy gap (90% → 75%) which is a great validation instinct — but I'd want to see whether the 75% used a truly held-out set of brands, or if brands could share pages/content types across the split.

Finding I'm choosing #2: "Myth 9 — Better Reader Engagement = Higher Rankings" (FALSE)

Claims a near-zero correlation (0.015) between reader engagement and position, but the paper itself flags that 85% of pages have no reader engagement data at all.

My methodology question: With 85% missing data, how was that correlation actually computed — were the missing rows dropped entirely, or imputed as zero engagement? If they were dropped, the correlation is only computed over the 15% of pages that do have GA4 tracking connected, which is a different (likely larger, more actively monitored) population than the whole portfolio — meaning "no relationship" might only hold within that non-random subset, not the full dataset the myth claims to test. This is exactly the kind of caveat my own w03 notebook flagged when 88% of my content_updated_date values were missing.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [ ]:
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import precision_score, recall_score, f1_score

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

df = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions_90d,
               SUM(gsc_clicks) AS clicks_90d,
               AVG(gsc_avg_position) AS avg_position,
               MAX(report_date) AS last_report_date
        FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT c.content_hash_id, a.client_hash_id, c.word_count,
           a.impressions_90d, a.clicks_90d, a.avg_position,
           DATE_DIFF('day', c.content_updated_date, a.last_report_date) AS days_since_last_update
    FROM read_parquet('{REL}/dim_content.parquet') c
    JOIN agg a ON c.content_hash_id = a.content_hash_id
""").df()

df['label'] = (df['impressions_90d'] < df['impressions_90d'].median()).astype(int)

# FIX: impressions_90d removed from features — it directly defines the label,
# so leaving it in would let the model "cheat" by reading the label's own source.
feature_cols = ['clicks_90d', 'avg_position', 'word_count', 'days_since_last_update']
model_df = df.dropna(subset=feature_cols + ['label', 'client_hash_id']).copy()

# --- BEFORE: naive random row split (ignores that many rows share a client) ---
X, y = model_df[feature_cols], model_df['label']
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
rf_before = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
pred_before = rf_before.predict(X_te)

# --- AFTER: honest grouped split by client ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))
train, test = model_df.iloc[train_idx], model_df.iloc[test_idx]
X_tr2, y_tr2 = train[feature_cols], train['label']
X_te2, y_te2 = test[feature_cols], test['label']
rf_after = RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
pred_after = rf_after.predict(X_te2)

summary = pd.DataFrame({
    'BEFORE (random split)': [precision_score(y_te, pred_before), recall_score(y_te, pred_before), f1_score(y_te, pred_before)],
    'AFTER (grouped split)': [precision_score(y_te2, pred_after), recall_score(y_te2, pred_after), f1_score(y_te2, pred_after)],
}, index=['precision', 'recall', 'f1'])
print(summary)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
model_df['leaky_col'] = model_df['label'] * 100 + np.random.normal(0, 1, len(model_df))
leak_feats = feature_cols + ['leaky_col']
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(model_df[leak_feats].fillna(0), model_df['label'], test_size=0.25, random_state=42)
rf_leak = RandomForestClassifier(n_estimators=100, random_state=42).fit(Xl_tr, yl_tr)
print("WITH LEAK accuracy:", (rf_leak.predict(Xl_te) == yl_te).mean())
print("HONEST (Section 2 AFTER) f1:", summary.loc['f1', 'AFTER (grouped split)'])

## 4. Claim rewrite
"Under a naive random split, the model showed an observed F1 of 0.37. However, once evaluated under an honest split — grouped by client, so no client's pages appeared in both training and test — performance dropped substantially to an F1 of 0.20 (precision 0.59, recall 0.12). This directional drop suggests the model is leaning partly on client-specific patterns rather than signals that generalize to genuinely new clients. The grouped-split number, not the random-split number, is the decision-support estimate that should inform any claim about how this model would perform on a brand-new client's content."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.